# ROC curves for individual miRNAs

This notebook evaluates each selected miRNA separately in the diagnostic and risk-stratification tasks using the internal hold-out test set.

In [ ]:
from itertools import cycle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import RocCurveDisplay

## File locations

In [ ]:
project_root = Path("..")
data_dir = project_root / "data" / "processed"
figure_dir = project_root / "results" / "figures"

figure_dir.mkdir(parents=True, exist_ok=True)

## Load the training and hold-out test data

In [ ]:
train_data = pd.read_table(
    data_dir / "train_exp_selected_mirs.txt",
    sep=" "
)

test_data = pd.read_table(
    data_dir / "test_exp_selected_mirs.txt",
    sep=" "
)

print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

## Prepare labels for CRC versus non-cancer

In [ ]:
diagnostic_train = train_data.iloc[:, :5].copy()
diagnostic_test = test_data.iloc[:, :5].copy()

diagnostic_train_labels = train_data.iloc[:, 24].copy()
diagnostic_train_labels = np.array([
    0 if stage == "Healthy" else 1
    for stage in diagnostic_train_labels
])

diagnostic_test_labels = test_data.iloc[:, 24].copy()
diagnostic_test_labels = np.array([
    0 if stage == "Healthy" else 1
    for stage in diagnostic_test_labels
])

print("Training CRC samples:", diagnostic_train_labels.sum())
print("Test CRC samples:", diagnostic_test_labels.sum())

## Prepare labels for low-risk versus high-risk CRC

In [ ]:
crc_train_data = train_data.loc[
    train_data.loc[:, "stage"] != "Healthy",
    :
].copy()

crc_train_expression = crc_train_data.iloc[:, :24].copy()
crc_train_stages = np.array(
    list(map(int, crc_train_data.loc[:, "stage"].copy()))
)

risk_train_groups = []

for stage in crc_train_stages:
    if stage in (0, 1, 2):
        risk_train_groups.append("LR")
    elif stage in (3, 4):
        risk_train_groups.append("HR")

risk_train_labels = np.array([
    0 if risk_group == "LR" else 1
    for risk_group in risk_train_groups
])

crc_test_data = test_data.loc[
    test_data.loc[:, "stage"] != "Healthy",
    :
].copy()

crc_test_expression = crc_test_data.iloc[:, :24].copy()
crc_test_stages = np.array(
    list(map(int, crc_test_data.loc[:, "stage"].copy()))
)

risk_test_groups = []

for stage in crc_test_stages:
    if stage in (0, 1, 2):
        risk_test_groups.append("LR")
    elif stage in (3, 4):
        risk_test_groups.append("HR")

risk_test_labels = np.array([
    0 if risk_group == "LR" else 1
    for risk_group in risk_test_groups
])

print("Training high-risk samples:", risk_train_labels.sum())
print("Test high-risk samples:", risk_test_labels.sum())

## Individual diagnostic-miRNA ROC curves

In [ ]:
fig, axis = plt.subplots(figsize=(10, 10))
plt.rcParams["font.family"] = "Times"

plot_colors = cycle([
    "orange",
    "cornflowerblue",
    "red",
    "navy",
    "goldenrod",
    "violet",
    "greenyellow",
    "gray",
    "black",
    "blue"
])

for mirna, color in zip(diagnostic_train.columns, plot_colors):
    single_train = diagnostic_train[[mirna]].to_numpy()
    single_test = diagnostic_test[[mirna]].to_numpy()

    mean_fpr = np.linspace(0, 1, 100)

    single_mirna_model = xgb.XGBClassifier()
    single_mirna_model.fit(single_train, diagnostic_train_labels)

    test_probabilities = single_mirna_model.predict_proba(
        single_test
    )[:, 1]

    roc_display = RocCurveDisplay.from_predictions(
        diagnostic_test_labels,
        test_probabilities,
        name="_",
        alpha=0,
        linewidth=0,
        ax=axis
    )

    interpolated_tpr = np.interp(
        mean_fpr,
        roc_display.fpr,
        roc_display.tpr
    )
    interpolated_tpr[0] = 0.0

    axis.plot(
        mean_fpr,
        interpolated_tpr,
        color=color,
        label=f"{mirna} (AUC={roc_display.roc_auc:.2f})",
        linewidth=2,
        alpha=0.8
    )

axis.plot([0, 1], [0, 1], "k--", linewidth=2.3)
axis.set_xlim([0.0, 1.0])
axis.set_ylim([0.0, 1.05])
axis.set_xlabel("False Positive Rate", fontsize=24)
axis.set_ylabel("True Positive Rate", fontsize=24)
axis.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    prop={"size": 16.5}
)
axis.tick_params(axis="both", which="major", labelsize=20)

plt.savefig(
    figure_dir / "single_miRNA_ROC_diagnostic.pdf",
    bbox_inches="tight"
)
plt.show()

## Individual risk-panel-miRNA ROC curves

In [ ]:
fig, axis = plt.subplots(figsize=(10, 10))
plt.rcParams["font.family"] = "Times"

plot_colors = cycle([
    "orange",
    "cornflowerblue",
    "red",
    "navy",
    "goldenrod",
    "violet",
    "greenyellow",
    "gray",
    "black",
    "blue",
    "cyan",
    "magenta",
    "lime",
    "teal",
    "brown",
    "pink",
    "darkgreen"
])

for mirna, color in zip(crc_train_expression.columns, plot_colors):
    single_train = crc_train_expression[[mirna]].to_numpy()
    single_test = crc_test_expression[[mirna]].to_numpy()

    mean_fpr = np.linspace(0, 1, 100)

    single_mirna_model = xgb.XGBClassifier()
    single_mirna_model.fit(single_train, risk_train_labels)

    test_probabilities = single_mirna_model.predict_proba(
        single_test
    )[:, 1]

    roc_display = RocCurveDisplay.from_predictions(
        risk_test_labels,
        test_probabilities,
        name="_",
        alpha=0,
        linewidth=0,
        ax=axis
    )

    interpolated_tpr = np.interp(
        mean_fpr,
        roc_display.fpr,
        roc_display.tpr
    )
    interpolated_tpr[0] = 0.0

    axis.plot(
        mean_fpr,
        interpolated_tpr,
        color=color,
        label=f"{mirna} (AUC={roc_display.roc_auc:.2f})",
        linewidth=2,
        alpha=0.8
    )

axis.plot([0, 1], [0, 1], "k--", linewidth=2.3)
axis.set_xlim([0.0, 1.0])
axis.set_ylim([0.0, 1.05])
axis.set_xlabel("False Positive Rate", fontsize=24)
axis.set_ylabel("True Positive Rate", fontsize=24)
axis.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    prop={"size": 16.5}
)
axis.tick_params(axis="both", which="major", labelsize=20)

plt.savefig(
    figure_dir / "single_miRNA_ROC_risk.pdf",
    bbox_inches="tight"
)
plt.show()